- benchmark.ipynb
- annotations
- videos

In [1]:
#### make folder
import os
import shutil

def make_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)
    else:
        shutil.rmtree(folder)
        os.makedirs(folder)
    return folder
  
#### extract number in file name
import re

def extract_frame_number(file_name):
    # Extract the frame number from the image file name, frame_00001.jpg -> 1
    return int(re.search(r'\d+', file_name).group())

def extract_alpha(file_name):
    return ''.join(filter(str.isalpha, file_name))

#### get file list in a folder
def get_file_list(input_folder, sort_rule, file_type1, file_type2=None):
    # Get the list of image files in the input folder
    if sort_rule == 'ending_number':
        return sorted([f for f in os.listdir(input_folder) if f.endswith(file_type1) or f.endswith(file_type2)], key=extract_frame_number)
    elif sort_rule == 'alpha':
        return sorted([f for f in os.listdir(input_folder) if f.endswith(file_type1) or f.endswith(file_type2)], key=extract_alpha)


#### video to images
import os
import shutil
import cv2

def make_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)
    else:
        shutil.rmtree(folder)
        os.makedirs(folder)
    return folder

def convert_video_to_images(input_video, output_folder):
    # Create output folder if it doesn't exist
    output_folder = make_folder(output_folder)

    # Open the video file
    video_capture = cv2.VideoCapture(input_video)
    success, frame = video_capture.read()
    count = 0

    # Read each frame and save it as an image
    while success: # and count < 1000:
        image_path = os.path.join(output_folder, f"{count:d}.jpg")  # Adjust the format as per your requirement
        cv2.imwrite(image_path, frame)  # Save the frame as an image
        success, frame = video_capture.read()  # Read next frame
        count += 1

    # Release the video capture object
    video_capture.release()
    return count #the number of frames saved (good/successful frames, not the whole frame count)

#### save into json
import json
def save(path, data):    
    # Save the dictionary to a JSON file
    with open(path, 'w') as json_file:
        json.dump(data, json_file)

#### load from json
import json
def load(path):
    # Load the JSON file
    with open(path, 'r') as json_file:
        return json.load(json_file)

In [2]:
anno_list = get_file_list('annotations', 'ending_number', '.mp4.json')
vid_list = get_file_list('videos', 'ending_number', '.mp4')

In [30]:
#frame no starting from 0, json idx  = frame no + 1
for i in range(0, len(vid_list)):
    anno = 'annotations/' + anno_list[i]
    video = vid_list[i]
    vid = 'videos/' + vid_list[i]
    anno_load = load(anno)
    frameno_list = list(anno_load.keys())
    frameno_list_int = [int(key) for key in frameno_list if int(key) != 0]
    frameno_list_int.sort()
    #print(frameno_list)
    count = convert_video_to_images(vid, 'temp/')
    
    source_folder = 'temp/'
    destination_folder = f'img/{vid_list[i]}/'
    make_folder(destination_folder)
    # Source and destination folders

    for j in frameno_list_int:
        # Image file to copy
        idx = int(j)-int(1)
        #print(idx)
        if idx < 0:
            continue
        if idx >= count:
            continue
        image_filename = f"{idx}.jpg"
        
        # Construct the full paths of the source and destination image files
        source_image_path = os.path.join(source_folder, image_filename)
        destination_image_path = os.path.join(destination_folder, image_filename)

        # Copy the image file from the source folder to the destination folder
        shutil.copy(source_image_path, destination_image_path)
        
        
        # Dictionary containing the data
        data_dict = anno_load[str(j)]

        # Specify the file path
        file_path = f"ground-truth/{video}_{idx}.txt"

        # Open the file in write mode and write the data line by line
        with open(file_path, 'w') as file:
            for i, (key, value) in enumerate(data_dict.items()):
                x1, y1, w, h = value
                value2 = [x1, y1, x1 + w, y1 + h]  # Convert to [x1, y1, x2, y2]
                line = f"person {' '.join(map(str, value2))}"  # Format the line without square brackets
                file.write(line)
                if i < len(data_dict) - 1:  # Add a newline if it's not the last line
                    file.write('\n')


In [29]:
count, idx

(402, 402)

In [ ]:
import os

# Specify the path of the directory for which you want to list the folders
directory_path = "img/"

# Get a list of all files and directories inside the specified directory
files_and_directories = os.listdir(directory_path)

# Filter only the directories from the list
folders = [folder for folder in files_and_directories if os.path.isdir(os.path.join(directory_path, folder))]

# Custom sorting function to extract the numeric part of the filename
def extract_number(filename):
    return int(re.search(r'\d+', filename).group())

# Sort the filenames based on the numeric part
sorted_folders = sorted(folders, key=extract_number)

for vid in sorted_folders:
    img_list = get_file_list('img/' + vid, 'ending_number', '.jpg')
    for img in img_list:
        index = int(re.search(r'\d+', img).group())
        #run model on img, get pred
        output_file = f'input/detection-results/{vid}_{index}.txt'
        print(output_file)

annotations into txt format

gt into txt format

videos: extract frames

# check data

In [3]:
for i in range(0, 1):
    anno = 'annotations/' + anno_list[i]
    video = vid_list[i]
    vid = 'videos/' + vid_list[i]
    anno_load = load(anno)
    frameno_list = list(anno_load.keys())
    frameno_list_int = [int(key) for key in frameno_list if int(key) != 0]
    frameno_list_int.sort()
    #print(frameno_list)
    convert_video_to_images(vid, 'temp/')

In [24]:
def draw_bboxes(frame, detections):
    for det in detections:
        x1,y1,x2,y2 = det
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 3) #(0,255,0), 3) green

from PIL import Image, ImageDraw
frame = Image.open('temp/24.jpg')
draw = ImageDraw.Draw(frame)

detections = list(anno_load['24'].values())
for i in detections:
    x1,y1,w,h = i
    draw.rectangle([x1,y1,x1+w,y1+h], outline="red", width=3)  # Draw a red rectangle

frame.save('test24.jpg')

# gt vs pred

In [34]:
import re

def extract_float(input_string):
    # Regular expression pattern to match floating-point numbers in the string
    pattern = r"\d+\.\d+"
    
    # Find all floating-point numbers in each line of the input string
    lines = input_string.strip().split("\n")
    coordinates_list = []
    
    for line in lines:
        numbers = re.findall(pattern, line)
        if len(numbers) >= 4:
            coordinates_list.append(list(map(float, numbers[-4:])))
    
    return coordinates_list

def extract_int(input_string):
    # Regular expression pattern to match floating-point numbers in the string
    pattern = r"\d+"
    
    # Find all floating-point numbers in each line of the input string
    lines = input_string.strip().split("\n")
    coordinates_list = []
    
    for line in lines:
        numbers = re.findall(pattern, line)
        if len(numbers) >= 4:
            coordinates_list.append(list(map(int, numbers[-4:])))
    
    return coordinates_list


# Example input string with coordinates
gt_string = """
person 282 977 503 1275
person -14 727 168 1088
person 1685 349 1742 549
person -32 523 169 951
person 1439 563 1663 1151
person 434 561 538 766
person 1259 382 1316 619
person 470 698 619 1059
person 249 631 367 960
person 86 801 320 1186
person 1340 372 1478 605
person 1066 366 1151 589
person 1150 370 1238 643
person 400 804 673 1182
person 1368 391 1459 569
person 377 583 467 788
person 520 571 684 947
person 604 710 722 1073
person 549 667 694 997
person 1289 364 1343 615
person 124 696 281 1069
person 1625 349 1694 556
person -57 888 179 1275
person 217 695 375 1077
person 955 380 1028 609
person 445 518 552 726
person 1627 598 1859 1125
person 1277 391 1388 599
person 516 587 632 795
person 430 533 542 723
person 363 610 526 892
person 184 764 357 1139
person 423 711 579 1073
person 333 664 533 987
person 316 439 448 751
person 469 791 676 1163
person 1280 391 1354 649
"""

pred_string = """
person 0.65665823 1.7578125 726.15234 161.60156 911.77734
person 0.6867903 98.4375 800.376 323.4375 1076.9678
person 0.6893056 399.2578 803.27637 676.9922 1021.333
person 0.7512834 2.9296875 523.125 172.5 811.0547
person 0.91086304 1624.3359 598.0078 1865.0391 1077.8906
person 0.9456007 1440.4102 566.63086 1666.4648 1077.627
"""

# Extract coordinates from the string and convert them to lists of lists
gt = extract_int(gt_string)
pred = extract_float(pred_string)

In [ ]:
#gt, pred

([[282, 977, 503, 1275],
  [14, 727, 168, 1088],
  [1685, 349, 1742, 549],
  [32, 523, 169, 951],
  [1439, 563, 1663, 1151],
  [434, 561, 538, 766],
  [1259, 382, 1316, 619],
  [470, 698, 619, 1059],
  [249, 631, 367, 960],
  [86, 801, 320, 1186],
  [1340, 372, 1478, 605],
  [1066, 366, 1151, 589],
  [1150, 370, 1238, 643],
  [400, 804, 673, 1182],
  [1368, 391, 1459, 569],
  [377, 583, 467, 788],
  [520, 571, 684, 947],
  [604, 710, 722, 1073],
  [549, 667, 694, 997],
  [1289, 364, 1343, 615],
  [124, 696, 281, 1069],
  [1625, 349, 1694, 556],
  [57, 888, 179, 1275],
  [217, 695, 375, 1077],
  [955, 380, 1028, 609],
  [445, 518, 552, 726],
  [1627, 598, 1859, 1125],
  [1277, 391, 1388, 599],
  [516, 587, 632, 795],
  [430, 533, 542, 723],
  [363, 610, 526, 892],
  [184, 764, 357, 1139],
  [423, 711, 579, 1073],
  [333, 664, 533, 987],
  [316, 439, 448, 751],
  [469, 791, 676, 1163],
  [1280, 391, 1354, 649]],
 [[1.7578125, 726.15234, 161.60156, 911.77734],
  [98.4375, 800.376, 323.437

In [36]:
from PIL import Image, ImageDraw
vid = '04'
img = '13'
frame = Image.open(f'/home/jupyter/test/jetson/img/uid_vid_000{vid}.mp4/{img}.jpg')
draw = ImageDraw.Draw(frame)

for i in gt:
    x1,y1,x2,y2 = i
    draw.rectangle([x1,y1,x2,y2], outline="red", width=3)  # Draw a red rectangle

frame.save(f'gt{vid}.jpg')

In [37]:
from PIL import Image, ImageDraw
frame = Image.open(f'/home/jupyter/test/jetson/img/uid_vid_000{vid}.mp4/{img}.jpg')
draw = ImageDraw.Draw(frame)

for i in pred:
    x1,y1,x2,y2 = i
    draw.rectangle([x1,y1,x2,y2], outline="red", width=3)  # Draw a red rectangle

frame.save(f'pred{vid}.jpg')

# speed

ft 960x960

In [5]:
import torch

from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

image_path = "/home/jupyter/test/jetson/4.jpg"
image = Image.open(image_path)

image_processor = RTDetrImageProcessor.from_pretrained("/home/jupyter/test/jetson/checkpoint-552")
model = RTDetrForObjectDetection.from_pretrained("/home/jupyter/test/jetson/checkpoint-552")

import time
# Record the start time
start_time = time.time()

inputs = image_processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

results = image_processor.post_process_object_detection(outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=0.3)

# Record the end time
end_time = time.time()
# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time} seconds")

for result in results:
    for score, label_id, box in zip(result["scores"], result["labels"], result["boxes"]):
        score, label = score.item(), label_id.item()
        box = [round(i, 2) for i in box.tolist()]
        print(f"{model.config.id2label[label]}: {score:.2f} {box}")


Elapsed time: 4.80638861656189 seconds
defect: 0.36 [400.02, 850.75, 443.2, 876.25]
defect: 0.34 [1044.06, 476.38, 1533.42, 763.81]
defect: 0.31 [1030.67, 288.14, 1533.38, 770.29]


In [8]:
import torch

from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

image_path = "/home/jupyter/test/jetson/4.jpg"
image = Image.open(image_path)

image_processor = RTDetrImageProcessor.from_pretrained("/home/jupyter/test/jetson/checkpoint-552")
model = RTDetrForObjectDetection.from_pretrained("/home/jupyter/test/jetson/checkpoint-552").to(device)
import time
# Record the start time
start_time = time.time()
for i in range(0,100):
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = image_processor.post_process_object_detection(outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=0.3)

# Record the end time
end_time = time.time()
# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time} seconds")

Elapsed time: 8.526227712631226 seconds


redetr pt 640x640

from huggingface_hub import snapshot_download
snapshot_download(repo_id="PekingU/rtdetr_r101vd_coco_o365", local_dir="/home/jupyter/rtdetr_101")

In [ ]:
import torch

from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

image_path = "/home/jupyter/test/jetson/4.jpg"
image = Image.open(image_path)

image_processor = RTDetrImageProcessor.from_pretrained("/home/jupyter/rtdetr_101")
model = RTDetrForObjectDetection.from_pretrained("/home/jupyter/rtdetr_101")

import time
# Record the start time
start_time = time.time()

inputs = image_processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

results = image_processor.post_process_object_detection(outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=0.3)

# Record the end time
end_time = time.time()
# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time} seconds")

for result in results:
    for score, label_id, box in zip(result["scores"], result["labels"], result["boxes"]):
        score, label = score.item(), label_id.item()
        box = [round(i, 2) for i in box.tolist()]
        #print(f"{model.config.id2label[label]}: {score:.2f} {box}")


Elapsed time: 0.5287952423095703 seconds
truck: 0.91 [1436.06, 125.49, 1823.87, 415.73]
bus: 0.91 [1105.84, 365.4, 1483.42, 640.07]
person: 0.90 [1034.98, 284.15, 1113.11, 477.25]
person: 0.89 [398.03, 608.62, 495.17, 792.88]
person: 0.89 [114.61, 746.39, 215.27, 975.37]
person: 0.88 [1333.46, 164.01, 1403.16, 327.49]
person: 0.82 [826.48, 373.55, 889.97, 548.02]
person: 0.81 [252.53, 610.45, 317.27, 814.32]
person: 0.80 [1150.91, 294.07, 1212.37, 462.67]
person: 0.78 [877.97, 399.51, 940.69, 523.42]
person: 0.76 [676.37, 520.88, 737.07, 642.51]
person: 0.74 [714.3, 480.15, 774.93, 611.0]
truck: 0.71 [155.0, 814.22, 653.69, 1078.95]
handbag: 0.70 [1093.29, 381.04, 1117.18, 448.19]
backpack: 0.65 [117.41, 777.99, 176.13, 864.07]
person: 0.64 [127.61, 595.39, 199.65, 772.75]
person: 0.64 [119.32, 403.57, 186.16, 607.21]
car: 0.61 [154.47, 813.82, 653.91, 1078.75]
person: 0.56 [297.14, 599.73, 355.75, 749.09]
person: 0.55 [248.15, 508.96, 299.45, 634.78]
handbag: 0.53 [1170.96, 321.4, 121

In [6]:
import torch

from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

image_path = "/home/jupyter/test/jetson/4.jpg"
image = Image.open(image_path)

image_processor = RTDetrImageProcessor.from_pretrained("/home/jupyter/rtdetr_101")
model = RTDetrForObjectDetection.from_pretrained("/home/jupyter/rtdetr_101").to(device)
import time
# Record the start time
start_time = time.time()
for i in range(0,100):
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = image_processor.post_process_object_detection(outputs, target_sizes=torch.tensor([image.size[::-1]]), threshold=0.3)

# Record the end time
end_time = time.time()
# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time} seconds")

Elapsed time: 7.178020000457764 seconds


ft 480x480

In [2]:
import torch

from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

# Check if CUDA is available, and set the device accordingly
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the image
image_path = "/home/jupyter/test/jetson/4.jpg"
image = Image.open(image_path)

# Load the image processor and model on the specified device
image_processor = RTDetrImageProcessor.from_pretrained("/home/jupyter/test/jetson/checkpoint-2000")
model = RTDetrForObjectDetection.from_pretrained("/home/jupyter/test/jetson/checkpoint-2000").to(device)

import time

# Record the start time
start_time = time.time()

for i in range(0, 100):
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = image_processor.post_process_object_detection(outputs, target_sizes=torch.tensor([image.size[::-1]]).to(device), threshold=0.3)

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time} seconds")

Elapsed time: 8.806694269180298 seconds


In [3]:
import torch

# Check the device where the model parameters are stored
device = next(model.parameters()).device

if device.type == 'cuda':
    print("Model is using CUDA (GPU) for processing.")
else:
    print("Model is using CPU for processing.")

Model is using CUDA (GPU) for processing.
